# Tavily + Oracle AI Database Demo
## Fresh Web Retrieval with Persistent Oracle Memory

This notebook demonstrates the Oracle-backed Tavily hybrid retrieval implementation in this repository. The integration turns Tavily search results into durable Oracle memory: Tavily retrieves fresh external knowledge, and Oracle stores the retrieved content, embeddings, raw payloads, and provenance so future AI workflows can reuse it.

The high-level idea is simple and powerful:

- Tavily provides fresh search results from the web.
- Oracle stores retrieved knowledge as persistent memory.
- Oracle enables cache reuse, semantic search, deduplication, JSON provenance, and vector indexing.
- Together, Tavily and Oracle form an AI-ready retrieval layer for RAG systems and agentic applications.

## Problem Statement

Tavily is useful when an application needs fresh external knowledge. It can retrieve current web results, source snippets, titles, URLs, and other structured metadata. However, many AI systems and agents repeatedly ask the same or semantically similar questions. If every request calls an external search API and discards the results afterward, the application loses useful context, spends more on repeated lookups, and has no durable memory of what it already learned.

Oracle solves that persistence problem by acting as a durable memory layer. Retrieved Tavily results can be embedded, written into Oracle, and later retrieved by semantic similarity or freshness-aware cache lookup. The system can search Oracle first, call Tavily only when fresh or relevant local memory is not enough, and then write new Tavily results back into Oracle.

**Tavily is the freshness layer; Oracle is the memory layer.**

In this repository, the Oracle path is implemented through `TavilyHybridClient` with `db_provider="oracle"`. The notebook uses the actual client options implemented here: `retrieval_mode="hybrid_search"`, `retrieval_mode="freshness_cache"`, `retrieval_mode="cache_then_memory"`, `persistence_depth`, memory thresholds, Oracle lifecycle metadata, URL/content-hash upsert, persistence limits, manual and automatic cache cleanup, convenience connection parameters, `enable_native_hybrid_search`, `enable_oracle_json_payload`, `enable_provenance_metadata`, `dedup_similarity_threshold`, and `ensure_oracle_vector_index()`.

## Architecture Overview

The integration follows an Oracle-first design. The application embeds the user query, checks Oracle memory, and then uses Tavily when it needs fresh external knowledge. When Tavily results are persisted, future queries can reuse Oracle memory instead of starting from an empty context window.

```text
User / Agent
   |
   v
TavilyHybridClient
   |
   v
Oracle-first lookup
   |
   +--> Cache hit / memory hit
   |        |
   |        v
   |     Return Oracle context
   |
   +--> Miss or refresh needed
            |
            v
        Tavily search
            |
            v
   Persist results into Oracle
            |
            v
   Future queries reuse Oracle memory
```

### Flow 1: Hybrid Search Mode

`retrieval_mode="hybrid_search"` searches Oracle local memory first. In this repository, it also calls Tavily whenever `max_foreign > 0`, merges local Oracle results with Tavily results, reranks the combined set with the configured `ranking_function`, and optionally persists Tavily results when `save_foreign=True`. Setting `max_foreign=0` performs an Oracle-only memory lookup.

When `enable_native_hybrid_search=True`, Oracle local retrieval combines vector candidates with Oracle Text candidates through `CONTAINS(...)` and `SCORE(1)`. If Oracle Text is not available or no Text index is created, vector search still works through `VECTOR_DISTANCE(...)`.

### Flow 2: Freshness Cache Mode

`retrieval_mode="freshness_cache"` uses Oracle as a freshness-aware cache. It searches Oracle rows inside the configured `cache_ttl_seconds` window and returns local results when at least one result meets `cache_score_threshold`. On a miss, it calls Tavily, optionally persists the new results, and returns the fresh Tavily results.

### Flow 3: Cache Then Memory Mode

`retrieval_mode="cache_then_memory"` checks fresh cache rows first. If there is no strong cache hit, it checks longer-lived Oracle memory rows. Only when both local layers miss does it call Tavily and optionally write through the new results. This is the closest notebook flow to "Tavily as freshness layer, Oracle as memory layer."

## Why Oracle?

Oracle adds durable, queryable, enterprise-grade memory to Tavily retrieval. The implementation in this repository uses Oracle as more than a simple table store: it can store vectors, run semantic similarity search, store JSON payloads, record provenance, apply metadata filters, create vector indexes, and skip near-duplicate inserts.

Oracle-specific value demonstrated in this notebook:

- Durable long-term memory for Tavily search results.
- Oracle `VECTOR` storage for embeddings and semantic similarity.
- Native JSON storage for raw Tavily payloads and provenance.
- Hybrid retrieval using vector search plus Oracle Text where available.
- Freshness-cache retrieval for repeated recent queries.
- Cache-then-memory retrieval for layered agent memory: fresh cache first, durable memory second, Tavily last.
- Vector index creation through `DBMS_VECTOR.CREATE_INDEX`, including HNSW/IVF options exposed by the client.
- Enterprise-ready persistence, auditability, and metadata filtering.
- URL/content-hash upsert through Oracle `MERGE` to avoid append-only growth.
- Persistence limits, score thresholds, and cleanup controls to keep memory manageable.
- Cleaner agent memory through semantic deduplication.

| Capability | Plain Tavily | Tavily + Oracle |
|-----------|--------------|-----------------|
| Fresh web search | Yes | Yes |
| Persistent memory | No | Yes |
| Semantic retrieval | Limited/session-based | Yes, via Oracle VECTOR |
| Metadata/provenance storage | Limited | Yes, via Oracle JSON and columns |
| Fresh cache reuse | No/limited | Yes, via `freshness_cache` |
| Cache + durable memory fallback | No | Yes, via `cache_then_memory` |
| Deduplication/upsert controls | No/limited | Yes |
| Cleanup/retention controls | No | Yes |
| Enterprise persistence | No | Yes |

## Setup and Prerequisites

This notebook expects a real Oracle-backed environment. It does not hardcode credentials and it does not create a non-Oracle fallback path.

Required prerequisites:

- A Python environment with this repository installed or available on `PYTHONPATH`.
- A Tavily API key in `TAVILY_API_KEY`.
- Oracle Database 23ai or a compatible Oracle Database with `VECTOR` support.
- `python-oracledb`, available through `python -m pip install -e ".[oracle]"` from this repository.
- An Oracle user/schema with permission to create or alter the demo table.
- Optional Oracle Text privileges if you want native hybrid vector + text retrieval.
- Optional Oracle wallet/mTLS configuration for Autonomous Database connections.

The default `TavilyHybridClient` embedding and reranking helpers use Cohere when no custom functions are supplied. To keep this notebook limited to Tavily + Oracle services, the code below provides a small deterministic local embedding function and a score-based ranking function through the client extension points. In production, replace those helpers with the same embedding and reranking models used by your AI application.

Required environment variables:

- `TAVILY_API_KEY`
- `ORACLE_USER`
- `ORACLE_PASSWORD`
- `ORACLE_DSN`

Optional environment variables used by the demo:

- `ORACLE_VECTOR_TABLE` defaults to `TAVILY_DOCUMENTS`
- `ORACLE_CONTENT_FIELD` defaults to `CONTENT`
- `ORACLE_EMBEDDINGS_FIELD` defaults to `EMBEDDINGS`
- `ORACLE_VECTOR_DIMENSION` defaults to `1024`; set it to match an existing table's VECTOR dimension
- `ORACLE_CACHE_TTL_SECONDS`, `ORACLE_CACHE_SCORE_THRESHOLD`, `ORACLE_MEMORY_SCORE_THRESHOLD`, and `ORACLE_MEMORY_MAX_RESULTS` tune cache and memory hits
- `ORACLE_UPSERT_KEY=source_url` or `ORACLE_UPSERT_KEY=content_hash` enables Oracle `MERGE` persistence
- `ORACLE_MAX_PERSISTED_FOREIGN` and `ORACLE_PERSIST_SCORE_THRESHOLD` limit what Tavily results are stored
- `ORACLE_AUTO_CLEANUP_CACHE=1` enables automatic expired-cache cleanup before search
- `ORACLE_CACHE_CLEANUP_INTERVAL_SECONDS` controls how often automatic cleanup can run
- `ORACLE_SYSDBA=1` to connect with `AUTH_MODE_SYSDBA` when needed locally
- `ORACLE_CONFIG_DIR`, `ORACLE_WALLET_LOCATION`, and `ORACLE_WALLET_PASSWORD` for wallet-based connections

In [3]:
import os
from pathlib import Path

env_path = Path("/Users/anishraj/Desktop/Projects/Pod4_demo/tavily-python/.env")
print("env exists:", env_path.exists())

loaded = []
with env_path.open(encoding="utf-8") as env_file:
    for raw_line in env_file:
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue

        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")

        if value:
            os.environ[key] = value
            loaded.append(key)

print("loaded keys:", loaded)

for name in ["TAVILY_API_KEY", "ORACLE_USER", "ORACLE_PASSWORD", "ORACLE_DSN"]:
    print(name, "set" if os.environ.get(name) else "missing")

env exists: True
loaded keys: ['TAVILY_API_KEY', 'ORACLE_USER', 'ORACLE_PASSWORD', 'ORACLE_DSN', 'ORACLE_CACHE_TTL_SECONDS', 'ORACLE_CACHE_SCORE_THRESHOLD', 'ORACLE_MEMORY_SCORE_THRESHOLD', 'ORACLE_MEMORY_MAX_RESULTS', 'ORACLE_MAX_PERSISTED_FOREIGN', 'ORACLE_PERSIST_SCORE_THRESHOLD', 'ORACLE_UPSERT_KEY']
TAVILY_API_KEY set
ORACLE_USER set
ORACLE_PASSWORD set
ORACLE_DSN set


In [4]:
import hashlib
import json
import math
import os
import re
from collections import Counter
from textwrap import shorten

try:
    import oracledb
except ImportError as exc:
    raise RuntimeError(
        "python-oracledb is required for this notebook. Install it with "
        "`python -m pip install -e \".[oracle]\"` or `python -m pip install oracledb`."
    ) from exc

try:
    from IPython.display import display as notebook_display
except ImportError:
    notebook_display = None

from tavily import TavilyHybridClient
from tavily.databases import oracledb as tavily_oracle_db

def load_local_env(paths=(".env", "../.env")):
    for path in paths:
        if not os.path.exists(path):
            continue
        with open(path, encoding="utf-8") as env_file:
            for raw_line in env_file:
                line = raw_line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, value = line.split("=", 1)
                key = key.strip()
                value = value.strip().strip('"').strip("'")
                if not value:
                    continue
                if key.startswith("ORACLE_") or key.upper().endswith("_PROXY"):
                    os.environ[key] = value
                else:
                    os.environ.setdefault(key, value)


load_local_env()

REQUIRED_ENV_VARS = [
    "TAVILY_API_KEY",
    "ORACLE_USER",
    "ORACLE_PASSWORD",
    "ORACLE_DSN",
]
missing = [name for name in REQUIRED_ENV_VARS if not os.environ.get(name)]
if missing:
    raise RuntimeError(
        "Missing required environment variables: "
        + ", ".join(missing)
        + ". Set them before running the notebook."
    )

TABLE_NAME = tavily_oracle_db.validate_identifier(
    os.environ.get("ORACLE_VECTOR_TABLE", "TAVILY_DOCUMENTS"),
    "ORACLE_VECTOR_TABLE",
)
CONTENT_FIELD = tavily_oracle_db.validate_identifier(
    os.environ.get("ORACLE_CONTENT_FIELD", "CONTENT"),
    "ORACLE_CONTENT_FIELD",
)
EMBEDDINGS_FIELD = tavily_oracle_db.validate_identifier(
    os.environ.get("ORACLE_EMBEDDINGS_FIELD", "EMBEDDINGS"),
    "ORACLE_EMBEDDINGS_FIELD",
)
CACHE_TIMESTAMP_FIELD = tavily_oracle_db.validate_identifier(
    os.environ.get("ORACLE_CACHE_TIMESTAMP_FIELD", "ADDED_AT"),
    "ORACLE_CACHE_TIMESTAMP_FIELD",
)
VECTOR_INDEX_NAME = tavily_oracle_db.validate_identifier(
    os.environ.get("ORACLE_VECTOR_INDEX_NAME", "TAVILY_DOCS_VEC_IDX"),
    "ORACLE_VECTOR_INDEX_NAME",
)
TEXT_INDEX_NAME = tavily_oracle_db.validate_identifier(
    os.environ.get("ORACLE_TEXT_INDEX_NAME", "TAVILY_DOCS_TEXT_IDX"),
    "ORACLE_TEXT_INDEX_NAME",
)

EMBEDDING_DIMENSION = int(os.environ.get("ORACLE_VECTOR_DIMENSION", "1024"))
if EMBEDDING_DIMENSION < 16:
    raise ValueError("ORACLE_VECTOR_DIMENSION must be at least 16 for this demo embedding function.")

FEATURES = {
    "oracle": 0,
    "database": 0,
    "vector": 1,
    "search": 2,
    "json": 3,
    "provenance": 4,
    "cache": 5,
    "freshness": 6,
    "tavily": 7,
    "agent": 8,
    "rag": 9,
    "memory": 10,
    "deduplication": 11,
    "index": 12,
    "ai": 13,
    "web": 14,
    "hybrid": 15,
}
SYNONYMS = {
    "23ai": "ai",
    "db": "database",
    "semantic": "vector",
    "similarity": "vector",
    "retrieval": "search",
    "retrieve": "search",
    "cached": "cache",
    "fresh": "freshness",
    "source": "provenance",
    "sources": "provenance",
    "dedup": "deduplication",
    "duplicate": "deduplication",
    "duplicates": "deduplication",
    "hnsw": "index",
    "ivf": "index",
    "agents": "agent",
}
TOKEN_RE = re.compile(r"[a-z0-9]+")


def demo_embedding_function(texts, input_type):
    """Small deterministic embedding helper for the demo notebook.

    The repository API accepts any embedding callable with this signature. This
    local helper keeps the notebook limited to Tavily + Oracle services while
    still exercising Oracle VECTOR storage, search, and deduplication.
    """
    vectors = []
    for text in texts:
        vector = [0.0] * EMBEDDING_DIMENSION
        for token in TOKEN_RE.findall(str(text).lower()):
            token = SYNONYMS.get(token, token)
            if token in FEATURES:
                vector[FEATURES[token] % EMBEDDING_DIMENSION] += 1.0
            else:
                digest = hashlib.sha256(token.encode("utf-8")).hexdigest()
                bucket = int(digest[:8], 16) % EMBEDDING_DIMENSION
                vector[bucket] += 0.15

        norm = math.sqrt(sum(value * value for value in vector))
        if norm == 0:
            vector[0] = 1.0
            norm = 1.0
        vectors.append([value / norm for value in vector])
    return vectors


def score_ranking_function(query, documents, top_n):
    """Rank the merged Oracle + Tavily result shape without external services."""
    return sorted(
        documents,
        key=lambda document: float(document.get("score") or 0.0),
        reverse=True,
    )[:top_n]


def summarize_results(results, title="Results", limit=5):
    print(title)
    print("-" * len(title))
    print(f"result_count={len(results)} origins={dict(Counter(result.get('origin') for result in results))}")
    for index, result in enumerate(results[:limit], start=1):
        score = result.get("score")
        score_text = f"{float(score):.4f}" if score is not None else "n/a"
        snippet = shorten(str(result.get("content", "")).replace("\n", " "), width=240, placeholder="...")
        url = result.get("url", "not included in hybrid result shape; inspect Oracle provenance")
        print(f"{index}. origin={result.get('origin')} score={score_text}")
        print(f"   url={url}")
        print(f"   snippet={snippet}")


def display_rows(rows, title=None):
    if title:
        print(title)
    if not rows:
        print("No rows returned.")
        return
    if notebook_display:
        notebook_display(rows)
    else:
        print(json.dumps(rows, default=str, indent=2))

print("Configuration loaded for table", TABLE_NAME)
print("Content column:", CONTENT_FIELD)
print("Embedding column:", EMBEDDINGS_FIELD)
print("Embedding dimension:", EMBEDDING_DIMENSION)

Configuration loaded for table TAVILY_DOCUMENTS
Content column: CONTENT
Embedding column: EMBEDDINGS
Embedding dimension: 1024


## Oracle Connection

The Oracle connection is the persistent memory backend for this demo. The setup cells create a normal `python-oracledb` connection so the notebook can create or alter the demo table before initializing `TavilyHybridClient`.

The SDK now supports both patterns:

- Advanced/application-owned connection: create `oracledb.connect(...)` yourself and pass `connection=connection`.
- Convenience connection: pass `oracle_user`, `oracle_password`, and `oracle_dsn` directly into `TavilyHybridClient` when you do not need custom connection lifecycle code.

Credentials are read from environment variables only. If you use Oracle Autonomous Database with wallet/mTLS, set the wallet-related environment variables shown below before running this cell. The password is never printed.

In [5]:
connection_args = {
    "user": os.environ["ORACLE_USER"],
    "password": os.environ["ORACLE_PASSWORD"],
    "dsn": os.environ["ORACLE_DSN"],
}

# Optional local administrative mode used by some Oracle Free / local setups.
if os.environ.get("ORACLE_SYSDBA") == "1":
    connection_args["mode"] = oracledb.AUTH_MODE_SYSDBA

# Optional wallet/mTLS settings for Autonomous Database environments.
OPTIONAL_CONNECTION_ENV = {
    "ORACLE_CONFIG_DIR": "config_dir",
    "ORACLE_WALLET_LOCATION": "wallet_location",
    "ORACLE_WALLET_PASSWORD": "wallet_password",
}
for env_name, arg_name in OPTIONAL_CONNECTION_ENV.items():
    value = os.environ.get(env_name)
    if value:
        connection_args[arg_name] = value

connection = oracledb.connect(**connection_args)

with connection.cursor() as cursor:
    cursor.execute("SELECT SYS_CONTEXT('USERENV', 'CURRENT_SCHEMA') FROM DUAL")
    current_schema = cursor.fetchone()[0]

print("Connected to Oracle schema:", current_schema)

Connected to Oracle schema: TAVILY_ITEST


## Schema / Table Setup

`TavilyHybridClient` expects an Oracle table with at least a content column and an embedding column. This demo also enables the optional Oracle AI/database features implemented in the repository, so the table includes timestamp, JSON, provenance, cache/memory lifecycle, and upsert columns.

Expected stored fields:

- Queryable text content in `CONTENT` or the configured content column.
- Embeddings in an Oracle `VECTOR` column.
- `ADDED_AT` timestamp for freshness-cache TTL checks.
- Raw Tavily payload in `RAW_PAYLOAD JSON` when `enable_oracle_json_payload=True`.
- Source URL, title, retrieval query, retrieval timestamp, retrieval mode, cache hit flag, insertion source, and provider name when `enable_provenance_metadata=True`.
- `MEMORY_SCOPE`, `EXPIRES_AT`, `LAST_SEEN_AT`, and `QUERY_COUNT` when `enable_oracle_memory_metadata=True`.
- `CONTENT_HASH` when `oracle_upsert_key="content_hash"` is enabled.

The setup is idempotent: it creates the table if missing and adds missing optional columns when the table already exists. The Oracle Text index is optional but enables the repository's native hybrid search SQL path: vector candidates plus `CONTAINS(...)` text candidates.

In [6]:
def oracle_error_code(exc):
    error_obj = exc.args[0] if exc.args else None
    return getattr(error_obj, "code", None)


def table_exists(connection, table_name):
    with connection.cursor() as cursor:
        cursor.execute(
            "SELECT COUNT(*) FROM USER_TABLES WHERE TABLE_NAME = :table_name",
            table_name=table_name,
        )
        return cursor.fetchone()[0] > 0


def index_exists(connection, index_name):
    with connection.cursor() as cursor:
        cursor.execute(
            "SELECT COUNT(*) FROM USER_INDEXES WHERE INDEX_NAME = :index_name",
            index_name=index_name,
        )
        return cursor.fetchone()[0] > 0


def existing_columns(connection, table_name):
    with connection.cursor() as cursor:
        cursor.execute(
            """
            SELECT COLUMN_NAME
            FROM USER_TAB_COLUMNS
            WHERE TABLE_NAME = :table_name
            """,
            table_name=table_name,
        )
        return {row[0] for row in cursor.fetchall()}


def ensure_column(connection, table_name, column_name, ddl):
    if column_name in existing_columns(connection, table_name):
        return False
    with connection.cursor() as cursor:
        cursor.execute(f"ALTER TABLE {table_name} ADD ({ddl})")
    connection.commit()
    return True


def ensure_demo_table(connection):
    if not table_exists(connection, TABLE_NAME):
        with connection.cursor() as cursor:
            cursor.execute(
                f"""
                CREATE TABLE {TABLE_NAME} (
                    ID NUMBER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
                    {CONTENT_FIELD} CLOB,
                    {EMBEDDINGS_FIELD} VECTOR({EMBEDDING_DIMENSION}, FLOAT32),
                    {CACHE_TIMESTAMP_FIELD} TIMESTAMP DEFAULT SYSTIMESTAMP,
                    RAW_PAYLOAD JSON,
                    SOURCE_URL VARCHAR2(1000),
                    SOURCE_TITLE VARCHAR2(500),
                    RETRIEVAL_QUERY VARCHAR2(1000),
                    RETRIEVAL_TIMESTAMP TIMESTAMP WITH TIME ZONE,
                    RETRIEVAL_MODE VARCHAR2(30),
                    CACHE_HIT NUMBER(1),
                    INSERTED_FROM VARCHAR2(30),
                    PROVIDER_NAME VARCHAR2(50),
                    MEMORY_SCOPE VARCHAR2(30),
                    EXPIRES_AT TIMESTAMP WITH TIME ZONE,
                    LAST_SEEN_AT TIMESTAMP WITH TIME ZONE,
                    QUERY_COUNT NUMBER DEFAULT 1,
                    CONTENT_HASH VARCHAR2(64)
                )
                """
            )
        connection.commit()
        print(f"Created table {TABLE_NAME}.")
        return

    column_definitions = {
        CONTENT_FIELD: f"{CONTENT_FIELD} CLOB",
        EMBEDDINGS_FIELD: f"{EMBEDDINGS_FIELD} VECTOR({EMBEDDING_DIMENSION}, FLOAT32)",
        CACHE_TIMESTAMP_FIELD: f"{CACHE_TIMESTAMP_FIELD} TIMESTAMP DEFAULT SYSTIMESTAMP",
        "RAW_PAYLOAD": "RAW_PAYLOAD JSON",
        "SOURCE_URL": "SOURCE_URL VARCHAR2(1000)",
        "SOURCE_TITLE": "SOURCE_TITLE VARCHAR2(500)",
        "RETRIEVAL_QUERY": "RETRIEVAL_QUERY VARCHAR2(1000)",
        "RETRIEVAL_TIMESTAMP": "RETRIEVAL_TIMESTAMP TIMESTAMP WITH TIME ZONE",
        "RETRIEVAL_MODE": "RETRIEVAL_MODE VARCHAR2(30)",
        "CACHE_HIT": "CACHE_HIT NUMBER(1)",
        "INSERTED_FROM": "INSERTED_FROM VARCHAR2(30)",
        "PROVIDER_NAME": "PROVIDER_NAME VARCHAR2(50)",
        "MEMORY_SCOPE": "MEMORY_SCOPE VARCHAR2(30)",
        "EXPIRES_AT": "EXPIRES_AT TIMESTAMP WITH TIME ZONE",
        "LAST_SEEN_AT": "LAST_SEEN_AT TIMESTAMP WITH TIME ZONE",
        "QUERY_COUNT": "QUERY_COUNT NUMBER DEFAULT 1",
        "CONTENT_HASH": "CONTENT_HASH VARCHAR2(64)",
    }

    added = []
    for column_name, ddl in column_definitions.items():
        if ensure_column(connection, TABLE_NAME, column_name, ddl):
            added.append(column_name)

    print(f"Table {TABLE_NAME} already exists.")
    if added:
        print("Added missing columns:", ", ".join(added))


ensure_demo_table(connection)

NATIVE_HYBRID_READY = False
if os.environ.get("ORACLE_CREATE_TEXT_INDEX", "1") == "1":
    try:
        if index_exists(connection, TEXT_INDEX_NAME):
            print(f"Oracle Text index {TEXT_INDEX_NAME} already exists.")
        else:
            with connection.cursor() as cursor:
                cursor.execute(
                    f"""
                    CREATE INDEX {TEXT_INDEX_NAME}
                    ON {TABLE_NAME}({CONTENT_FIELD})
                    INDEXTYPE IS CTXSYS.CONTEXT
                    """
                )
            connection.commit()
            print(f"Created Oracle Text index {TEXT_INDEX_NAME}.")
        NATIVE_HYBRID_READY = True
    except oracledb.DatabaseError as exc:
        print("Oracle Text index was not created. Native vector search still works.")
        print("Oracle error code:", oracle_error_code(exc))
        print("Set ORACLE_CREATE_TEXT_INDEX=0 to skip this step in restricted schemas.")
else:
    print("Skipping Oracle Text index creation because ORACLE_CREATE_TEXT_INDEX=0.")

print("Native Oracle hybrid search enabled for this run:", NATIVE_HYBRID_READY)

Table TAVILY_DOCUMENTS already exists.
Oracle Text index TAVILY_DOCS_TEXT_IDX already exists.
Native Oracle hybrid search enabled for this run: True


## Initialize the Tavily + Oracle Client

`TavilyHybridClient` is the main integration point. It wraps Tavily search and uses Oracle for retrieval and persistence.

This notebook creates three Oracle-only clients with the same table and connection:

- `hybrid_client` uses `retrieval_mode="hybrid_search"` for semantic memory retrieval plus Tavily results.
- `cache_client` uses `retrieval_mode="freshness_cache"` for TTL-aware Oracle cache reuse.
- `cache_memory_client` uses `retrieval_mode="cache_then_memory"` to check fresh cache, then durable Oracle memory, then Tavily.

The clients enable JSON payload storage, provenance metadata, cache/memory lifecycle metadata, semantic deduplication, persistence limits, optional upsert, optional automatic cleanup, and optional vector index configuration. The deterministic embedding and ranking functions are passed through the public constructor parameters so the notebook can run without any embedding service beyond Tavily and Oracle.

The code still passes an existing `connection` because earlier notebook cells need that connection for schema setup. In an application that already has the schema, you can omit `connection` and instead pass `oracle_user`, `oracle_password`, `oracle_dsn`, and optional `oracle_connection_kwargs`.

In [7]:
CACHE_TTL_SECONDS = int(os.environ.get("ORACLE_CACHE_TTL_SECONDS", "3600"))
CACHE_SCORE_THRESHOLD = float(os.environ.get("ORACLE_CACHE_SCORE_THRESHOLD", "0.05"))
MEMORY_SCORE_THRESHOLD = float(os.environ.get("ORACLE_MEMORY_SCORE_THRESHOLD", "0.05"))
MEMORY_MAX_RESULTS = int(os.environ.get("ORACLE_MEMORY_MAX_RESULTS", str(MAX_LOCAL if "MAX_LOCAL" in globals() else 5)))
DEDUP_SIMILARITY_THRESHOLD = float(os.environ.get("ORACLE_DEDUP_SIMILARITY_THRESHOLD", "0.95"))

MAX_PERSISTED_FOREIGN_ENV = os.environ.get("ORACLE_MAX_PERSISTED_FOREIGN")
MAX_PERSISTED_FOREIGN = int(MAX_PERSISTED_FOREIGN_ENV) if MAX_PERSISTED_FOREIGN_ENV else None
PERSIST_SCORE_THRESHOLD_ENV = os.environ.get("ORACLE_PERSIST_SCORE_THRESHOLD")
PERSIST_SCORE_THRESHOLD = float(PERSIST_SCORE_THRESHOLD_ENV) if PERSIST_SCORE_THRESHOLD_ENV else None
ORACLE_UPSERT_KEY = os.environ.get("ORACLE_UPSERT_KEY") or None
AUTO_CLEANUP_CACHE = os.environ.get("ORACLE_AUTO_CLEANUP_CACHE", "0") == "1"
CACHE_CLEANUP_INTERVAL_SECONDS = int(os.environ.get("ORACLE_CACHE_CLEANUP_INTERVAL_SECONDS", "3600"))

COMMON_CLIENT_CONFIG = {
    "api_key": os.environ["TAVILY_API_KEY"],
    "db_provider": "oracle",
    "connection": connection,
    "table_name": TABLE_NAME,
    "embeddings_field": EMBEDDINGS_FIELD,
    "content_field": CONTENT_FIELD,
    "cache_timestamp_field": CACHE_TIMESTAMP_FIELD,
    "embedding_function": demo_embedding_function,
    "ranking_function": score_ranking_function,
    "enable_native_hybrid_search": NATIVE_HYBRID_READY,
    "oracle_metadata_filters": {"provider_name": "tavily"},
    "enable_oracle_json_payload": True,
    "enable_provenance_metadata": True,
    "enable_oracle_memory_metadata": True,
    "oracle_upsert_key": ORACLE_UPSERT_KEY,
    "max_persisted_foreign": MAX_PERSISTED_FOREIGN,
    "persist_score_threshold": PERSIST_SCORE_THRESHOLD,
    "auto_cleanup_cache": AUTO_CLEANUP_CACHE,
    "cache_cleanup_interval_seconds": CACHE_CLEANUP_INTERVAL_SECONDS,
    "vector_index_name": VECTOR_INDEX_NAME,
    "vector_index_type": os.environ.get("ORACLE_VECTOR_INDEX_TYPE", "HNSW"),
    "vector_index_distance": os.environ.get("ORACLE_VECTOR_INDEX_DISTANCE", "COSINE"),
    "dedup_similarity_threshold": DEDUP_SIMILARITY_THRESHOLD,
}

hybrid_client = TavilyHybridClient(
    **COMMON_CLIENT_CONFIG,
    retrieval_mode="hybrid_search",
    persistence_depth="cache_plus_memory",
)

cache_client = TavilyHybridClient(
    **COMMON_CLIENT_CONFIG,
    retrieval_mode="freshness_cache",
    cache_ttl_seconds=CACHE_TTL_SECONDS,
    cache_score_threshold=CACHE_SCORE_THRESHOLD,
    persistence_depth="cache_only",
)

cache_memory_client = TavilyHybridClient(
    **COMMON_CLIENT_CONFIG,
    retrieval_mode="cache_then_memory",
    cache_ttl_seconds=CACHE_TTL_SECONDS,
    cache_score_threshold=CACHE_SCORE_THRESHOLD,
    memory_score_threshold=MEMORY_SCORE_THRESHOLD,
    memory_max_results=MEMORY_MAX_RESULTS,
    persistence_depth="cache_plus_memory",
)

print("Initialized TavilyHybridClient for Oracle hybrid search.")
print("Initialized TavilyHybridClient for Oracle freshness cache.")
print("Initialized TavilyHybridClient for Oracle cache-then-memory.")
print("Oracle upsert key:", ORACLE_UPSERT_KEY or "disabled")
print("Automatic cleanup:", AUTO_CLEANUP_CACHE, "interval_seconds=", CACHE_CLEANUP_INTERVAL_SECONDS)

Initialized TavilyHybridClient for Oracle hybrid search.
Initialized TavilyHybridClient for Oracle freshness cache.
Initialized TavilyHybridClient for Oracle cache-then-memory.
Oracle upsert key: source_url
Automatic cleanup: False interval_seconds= 3600


## Demo Query Setup

The demo uses stable Oracle/Tavily retrieval questions that exercise web freshness, Oracle cache reuse, durable Oracle memory, and semantic similarity. The same query shape is used across all three retrieval modes so reviewers can see how results move from Tavily retrieval into Oracle-backed reuse.

The three modes shown later are:

- `hybrid_search`: Oracle memory + Tavily results, then rerank.
- `freshness_cache`: fresh Oracle cache first, Tavily on cache miss.
- `cache_then_memory`: fresh Oracle cache first, durable Oracle memory second, Tavily only when both miss.

The search parameters are intentionally small for a notebook demo:

- `max_results=5` keeps output readable.
- `max_local=5` asks Oracle for up to five local cache/memory rows.
- `max_foreign=5` allows Tavily to supply fresh web results when the mode calls for external retrieval.
- `search_depth="advanced"` asks Tavily for stronger web retrieval while preserving the same SDK API used elsewhere in the repository.

In [8]:
DEMO_QUERY = "Virat Kohli cricket career records"
SIMILAR_QUERY = "Indian cricket batsman Virat Kohli achievements"
CACHE_QUERY = "Virat Kohli latest cricket news"

MAX_RESULTS = 5
MAX_LOCAL = 5
MAX_FOREIGN = 5
TAVILY_SEARCH_OPTIONS = {
    "search_depth": "advanced",
    "include_answer": False,
}

print("Hybrid query:", DEMO_QUERY)
print("Semantic memory query:", SIMILAR_QUERY)
print("Cache query:", CACHE_QUERY)

Hybrid query: Virat Kohli cricket career records
Semantic memory query: Indian cricket batsman Virat Kohli achievements
Cache query: Virat Kohli latest cricket news


# Mode 1: Hybrid Search

Hybrid search is the long-term memory flow. The client embeds the query, searches Oracle memory with `VECTOR_DISTANCE(...)`, optionally adds Oracle Text scoring when `enable_native_hybrid_search=True`, calls Tavily when `max_foreign > 0`, merges the local and foreign result sets, reranks them, and persists Tavily results when `save_foreign=True`.

Why this matters:

- Oracle can return semantically relevant prior results even when the wording changes.
- Tavily can still contribute fresh web results.
- `save_foreign=True` turns one-time Tavily retrieval into durable Oracle memory.
- JSON payload and provenance fields preserve source details that the compact hybrid result shape does not return directly.

Expected output:

- `origin="local"` means the result came from Oracle memory.
- `origin="foreign"` means the result came from Tavily during this call.
- Scores come from Oracle vector/text retrieval for local results and Tavily for foreign results, then the configured ranking function sorts the merged list.

In [9]:
hybrid_results = hybrid_client.search(
    DEMO_QUERY,
    max_results=MAX_RESULTS,
    max_local=MAX_LOCAL,
    max_foreign=MAX_FOREIGN,
    save_foreign=True,
    **TAVILY_SEARCH_OPTIONS,
)

summarize_results(hybrid_results, title="Hybrid search results")

Hybrid search results
---------------------
result_count=5 origins={'foreign': 5}
1. origin=foreign score=0.9999
   url=not included in hybrid result shape; inspect Oracle provenance
   snippet=7. What are Virat Kohli’s records as captain of India? Virat Kohli is one of India’s most successful cricket captains. Major captaincy records include: • Most Test wins as Indian captain (40+) • Led India to No.1 position in ICC Test...
2. origin=foreign score=0.9998
   url=not included in hybrid result shape; inspect Oracle provenance
   snippet=Kohli holds several cricket records, including the most individual hundreds in ODI matches and the most runs scored in a single edition of an ODI World Cup. He has been named Player of the Tournament at global events on three occasions:...
3. origin=foreign score=0.9996
   url=not included in hybrid result shape; inspect Oracle provenance
   snippet=Kohli's junior cricket career kicked off in October 2002 at the Luhnu Cricket Ground against Himachal Pra

The hybrid result list shows the repository's common result shape: content, score, and origin. URLs and titles from Tavily are intentionally persisted into Oracle provenance fields when `enable_oracle_json_payload=True` and `enable_provenance_metadata=True` are enabled. The next section queries Oracle directly to show that the fresh Tavily retrieval has become durable memory.

If the table was empty before this run, the first hybrid call should include Tavily results and write them into Oracle. If the table already contained relevant memory, local Oracle results may also appear in the merged result set.

## Demonstrate Oracle Persistence

After Tavily fetches fresh results, `save_foreign=True` writes those results into Oracle through the repository's Oracle insert path. This turns a one-time web search into reusable memory for future RAG and agent workflows.

This query inspects the durable rows without exposing secrets. It selects the original retrieval query, source URL, retrieval timestamp, content snippet, provider name, and JSON provenance fields.

In [10]:
with connection.cursor() as cursor:
    cursor.execute(
        f"""
        SELECT RETRIEVAL_QUERY,
               SOURCE_URL,
               SOURCE_TITLE,
               RETRIEVAL_TIMESTAMP,
               PROVIDER_NAME,
               DBMS_LOB.SUBSTR({CONTENT_FIELD}, 220, 1) AS CONTENT_SNIPPET,
               JSON_VALUE(RAW_PAYLOAD, '$.provenance.retrieval_mode') AS RAW_RETRIEVAL_MODE
        FROM {TABLE_NAME}
        WHERE PROVIDER_NAME = :provider_name
        ORDER BY RETRIEVAL_TIMESTAMP DESC NULLS LAST
        FETCH FIRST 5 ROWS ONLY
        """,
        provider_name="tavily",
    )
    persistence_rows = [
        {
            "retrieval_query": row[0],
            "source_url": row[1],
            "source_title": row[2],
            "retrieval_timestamp": row[3],
            "provider_name": row[4],
            "content_snippet": row[5],
            "raw_retrieval_mode": row[6],
        }
        for row in cursor.fetchall()
    ]

display_rows(persistence_rows, title="Recent Oracle memory rows")

Recent Oracle memory rows


[{'retrieval_query': 'Virat Kohli cricket career records',
  'source_url': 'https://en.wikipedia.org/wiki/Virat_Kohli',
  'source_title': 'Virat Kohli',
  'retrieval_timestamp': datetime.datetime(2026, 6, 1, 22, 25, 45),
  'provider_name': 'tavily',
  'content_snippet': 'Kohli\'s junior cricket career kicked off in October 2002 at the Luhnu Cricket Ground against Himachal Pradesh. His first half-century in domestic cricket happened at Feroze Shah Kotla, where he scored 70 runs "Run (crick',
  'raw_retrieval_mode': 'hybrid_search'},
 {'retrieval_query': 'Virat Kohli cricket career records',
  'source_url': 'https://www.britannica.com/biography/Virat-Kohli',
  'source_title': 'Virat Kohli | Life, Career, Cricket, Awards, T20I and Test ... - Britannica',
  'retrieval_timestamp': datetime.datetime(2026, 6, 1, 22, 25, 45),
  'provider_name': 'tavily',
  'content_snippet': 'Kohli holds several cricket records, including the most individual hundreds in ODI matches and the most runs scored in 

## Demonstrate Oracle VECTOR Search

Oracle `VECTOR` enables semantic memory. A future user or agent can ask a related question with different wording and retrieve prior Tavily results from Oracle without making a fresh Tavily call.

This cell forces an Oracle-only memory lookup by setting `max_foreign=0`. That uses the same `TavilyHybridClient.search(...)` API, but prevents the Tavily fallback portion of hybrid mode. If relevant rows were persisted earlier, the results should have `origin="local"`.

What this demonstrates:

- Query text is embedded by the configured embedding function.
- Oracle compares the query vector with stored result vectors through `VECTOR_DISTANCE(...)`.
- Similar wording can recall persisted memory even when the query is not identical.

In [11]:
semantic_memory_results = hybrid_client.search(
    SIMILAR_QUERY,
    max_results=MAX_RESULTS,
    max_local=MAX_LOCAL,
    max_foreign=0,
    save_foreign=False,
)

summarize_results(semantic_memory_results, title="Oracle-only VECTOR memory lookup")

Oracle-only VECTOR memory lookup
--------------------------------
result_count=5 origins={'local': 5}
1. origin=local score=0.3419
   url=not included in hybrid result shape; inspect Oracle provenance
   snippet=7. What are Virat Kohli’s records as captain of India? Virat Kohli is one of India’s most successful cricket captains. Major captaincy records include: • Most Test wins as Indian captain (40+) • Led India to No.1 position in ICC Test...
2. origin=local score=0.1868
   url=not included in hybrid result shape; inspect Oracle provenance
   snippet=Kohli holds several cricket records, including the most individual hundreds in ODI matches and the most runs scored in a single edition of an ODI World Cup. He has been named Player of the Tournament at global events on three occasions:...
3. origin=local score=0.1315
   url=not included in hybrid result shape; inspect Oracle provenance
   snippet=Kohli's junior cricket career kicked off in October 2002 at the Luhnu Cricket Ground agains

## Demonstrate Native JSON / Provenance

Tavily returns structured result metadata. With `enable_oracle_json_payload=True`, the repository stores the raw Tavily result and a provenance object in `RAW_PAYLOAD JSON`. With `enable_provenance_metadata=True`, it also writes source URL, source title, retrieval query, retrieval timestamp, retrieval mode, cache-hit flag, insertion source, and provider name into first-class columns.

This is important for enterprise AI applications because agents need source traceability. Provenance lets the application audit which search query produced a memory row, which provider supplied it, when it was retrieved, and which URL should be cited.

In [12]:
with connection.cursor() as cursor:
    cursor.execute(
        f"""
        SELECT SOURCE_URL,
               SOURCE_TITLE,
               RETRIEVAL_QUERY,
               RETRIEVAL_TIMESTAMP,
               RETRIEVAL_MODE,
               CACHE_HIT,
               INSERTED_FROM,
               PROVIDER_NAME,
               JSON_VALUE(RAW_PAYLOAD, '$.provenance.provider_name') AS RAW_PROVIDER,
               JSON_VALUE(RAW_PAYLOAD, '$.result.url') AS RAW_URL
        FROM {TABLE_NAME}
        WHERE JSON_EXISTS(RAW_PAYLOAD, '$.provenance.provider_name')
        ORDER BY RETRIEVAL_TIMESTAMP DESC NULLS LAST
        FETCH FIRST 5 ROWS ONLY
        """
    )
    provenance_rows = [
        {
            "source_url": row[0],
            "source_title": row[1],
            "retrieval_query": row[2],
            "retrieval_timestamp": row[3],
            "retrieval_mode": row[4],
            "cache_hit": row[5],
            "inserted_from": row[6],
            "provider_name": row[7],
            "raw_provider": row[8],
            "raw_url": row[9],
        }
        for row in cursor.fetchall()
    ]

display_rows(provenance_rows, title="Oracle JSON and provenance rows")

Oracle JSON and provenance rows


[{'source_url': 'https://en.wikipedia.org/wiki/Virat_Kohli',
  'source_title': 'Virat Kohli',
  'retrieval_query': 'Virat Kohli cricket career records',
  'retrieval_timestamp': datetime.datetime(2026, 6, 1, 22, 25, 45),
  'retrieval_mode': 'hybrid_search',
  'cache_hit': 0,
  'inserted_from': 'tavily',
  'provider_name': 'tavily',
  'raw_provider': 'tavily',
  'raw_url': 'https://en.wikipedia.org/wiki/Virat_Kohli'},
 {'source_url': 'https://www.britannica.com/biography/Virat-Kohli',
  'source_title': 'Virat Kohli | Life, Career, Cricket, Awards, T20I and Test ... - Britannica',
  'retrieval_query': 'Virat Kohli cricket career records',
  'retrieval_timestamp': datetime.datetime(2026, 6, 1, 22, 25, 45),
  'retrieval_mode': 'hybrid_search',
  'cache_hit': 0,
  'inserted_from': 'tavily',
  'provider_name': 'tavily',
  'raw_provider': 'tavily',
  'raw_url': 'https://www.britannica.com/biography/Virat-Kohli'},
 {'source_url': 'https://www.vedantu.com/general-knowledge/virat-kohli-world-rec

## Demonstrate Semantic Deduplication

Search systems can easily store duplicate or near-duplicate content when an agent repeats a question. The Oracle implementation supports semantic deduplication through `dedup_similarity_threshold`. Before inserting a new Tavily result, the client checks the nearest stored vector in Oracle. If similarity is at or above the configured threshold, the insert is skipped.

This keeps Oracle memory clean:

- Repeated searches do not endlessly duplicate the same source content.
- Agents can reuse memory without bloating the table.
- Deduplication uses Oracle `VECTOR_DISTANCE(...)`, so it works on semantic similarity rather than exact string equality.

The cell below runs the same query again with `save_foreign=True` and compares row counts before and after. If Tavily returns the same or semantically very similar content, the count should stay the same. If Tavily returns genuinely new content below the threshold, new rows may be inserted.

In [13]:
def tavily_provider_row_count(connection):
    with connection.cursor() as cursor:
        cursor.execute(
            f"SELECT COUNT(*) FROM {TABLE_NAME} WHERE PROVIDER_NAME = :provider_name",
            provider_name="tavily",
        )
        return cursor.fetchone()[0]


rows_before_dedup = tavily_provider_row_count(connection)

dedup_results = hybrid_client.search(
    DEMO_QUERY,
    max_results=MAX_RESULTS,
    max_local=0,
    max_foreign=MAX_FOREIGN,
    save_foreign=True,
    **TAVILY_SEARCH_OPTIONS,
)

rows_after_dedup = tavily_provider_row_count(connection)
rows_inserted = rows_after_dedup - rows_before_dedup

summarize_results(dedup_results, title="Repeated Tavily search with Oracle semantic deduplication")
print("Rows before:", rows_before_dedup)
print("Rows after:", rows_after_dedup)
print("Rows inserted:", rows_inserted)
print("Dedup threshold:", DEDUP_SIMILARITY_THRESHOLD)

if rows_inserted == 0:
    print("No new rows were inserted; Oracle semantic deduplication skipped near-duplicates.")
else:
    print("New rows were inserted; Tavily returned content below the configured dedup threshold.")

Repeated Tavily search with Oracle semantic deduplication
---------------------------------------------------------
result_count=5 origins={'foreign': 5}
1. origin=foreign score=0.9999
   url=not included in hybrid result shape; inspect Oracle provenance
   snippet=7. What are Virat Kohli’s records as captain of India? Virat Kohli is one of India’s most successful cricket captains. Major captaincy records include: • Most Test wins as Indian captain (40+) • Led India to No.1 position in ICC Test...
2. origin=foreign score=0.9998
   url=not included in hybrid result shape; inspect Oracle provenance
   snippet=Kohli holds several cricket records, including the most individual hundreds in ODI matches and the most runs scored in a single edition of an ODI World Cup. He has been named Player of the Tournament at global events on three occasions:...
3. origin=foreign score=0.9996
   url=not included in hybrid result shape; inspect Oracle provenance
   snippet=Kohli's junior cricket career kic

## Demonstrate Vector Index Creation / Usage

Vector indexes improve semantic search performance as Oracle memory grows. The repository exposes `ensure_oracle_vector_index()` on `TavilyHybridClient`. The helper checks `USER_INDEXES` first and calls `DBMS_VECTOR.CREATE_INDEX` only when the requested index is missing.

Supported index options are exposed by constructor parameters:

- `vector_index_type="HNSW"` or `"IVF"`
- `vector_index_distance="COSINE"` or another supported Oracle metric
- HNSW parameters such as `vector_index_neighbors` and `vector_index_efconstruction`
- IVF parameters such as `vector_index_partitions`

This step is optional for small demos, but it matters for larger AI memory stores. If your Oracle user does not have the required vector index privileges, the cell will report the Oracle error and the rest of the notebook can still demonstrate table-backed vector search.

In [14]:
try:
    created_vector_index = hybrid_client.ensure_oracle_vector_index()
    if created_vector_index:
        print(f"Created Oracle vector index {VECTOR_INDEX_NAME}.")
    else:
        print(f"Oracle vector index {VECTOR_INDEX_NAME} already exists.")
except oracledb.DatabaseError as exc:
    print("Vector index was not created. Oracle VECTOR search can still run without this optional index.")
    print("Oracle error code:", oracle_error_code(exc))
    print("Check DBMS_VECTOR privileges and Oracle VECTOR index support for this schema.")

with connection.cursor() as cursor:
    cursor.execute(
        """
        SELECT INDEX_NAME, INDEX_TYPE, STATUS
        FROM USER_INDEXES
        WHERE INDEX_NAME IN (:vector_index_name, :text_index_name)
        ORDER BY INDEX_NAME
        """,
        vector_index_name=VECTOR_INDEX_NAME,
        text_index_name=TEXT_INDEX_NAME,
    )
    index_rows = [
        {"index_name": row[0], "index_type": row[1], "status": row[2]}
        for row in cursor.fetchall()
    ]

display_rows(index_rows, title="Oracle indexes visible to this schema")

Vector index was not created. Oracle VECTOR search can still run without this optional index.
Oracle error code: 51962
Check DBMS_VECTOR privileges and Oracle VECTOR index support for this schema.
Oracle indexes visible to this schema


[{'index_name': 'TAVILY_DOCS_TEXT_IDX',
  'index_type': 'DOMAIN',
  'status': 'VALID'}]

# Mode 2: Freshness Cache Search

`retrieval_mode="freshness_cache"` uses Oracle as a freshness-aware cache. The client searches rows whose timestamp is inside `cache_ttl_seconds`. If at least one local result reaches `cache_score_threshold`, the method returns Oracle rows and skips Tavily. If the cache is missing, expired, or below the threshold, it calls Tavily and optionally writes the fresh results back into Oracle.

What this step demonstrates:

- Oracle can behave like a persistent freshness cache.
- TTL is enforced with the configured timestamp column, defaulting to `ADDED_AT`.
- A second call can reuse fresh Oracle rows instead of making a repeated Tavily call.
- The same provenance, lifecycle metadata, persistence controls, and deduplication settings apply to cache miss persistence.

This is different from `cache_then_memory`: freshness-cache mode has only one local layer, the fresh cache. Cache-then-memory has two local layers, fresh cache and durable memory.

The first call may be a cache hit if you have already run this notebook or the table already contains fresh similar rows. The second call should be a cache hit when fresh persisted rows meet the threshold.

In [15]:
cache_first_results = cache_client.search(
    CACHE_QUERY,
    max_results=MAX_RESULTS,
    max_local=MAX_LOCAL,
    max_foreign=MAX_FOREIGN,
    save_foreign=True,
    **TAVILY_SEARCH_OPTIONS,
)

summarize_results(cache_first_results, title="Freshness-cache first call")
first_origins = Counter(result.get("origin") for result in cache_first_results)
print("First call behavior:", "cache hit" if first_origins.get("local") else "cache miss with Tavily fallback")

cache_second_results = cache_client.search(
    CACHE_QUERY,
    max_results=MAX_RESULTS,
    max_local=MAX_LOCAL,
    max_foreign=MAX_FOREIGN,
    save_foreign=True,
    **TAVILY_SEARCH_OPTIONS,
)

summarize_results(cache_second_results, title="Freshness-cache second call")
second_origins = Counter(result.get("origin") for result in cache_second_results)
print("Second call behavior:", "cache hit" if second_origins.get("local") else "cache miss with Tavily fallback")
print("TTL seconds:", CACHE_TTL_SECONDS)
print("Cache score threshold:", CACHE_SCORE_THRESHOLD)

Freshness-cache first call
--------------------------
result_count=3 origins={'local': 3}
1. origin=local score=0.3518
   url=not included in hybrid result shape; inspect Oracle provenance
   snippet=7. What are Virat Kohli’s records as captain of India? Virat Kohli is one of India’s most successful cricket captains. Major captaincy records include: • Most Test wins as Indian captain (40+) • Led India to No.1 position in ICC Test...
2. origin=local score=0.1851
   url=not included in hybrid result shape; inspect Oracle provenance
   snippet=Kohli holds several cricket records, including the most individual hundreds in ODI matches and the most runs scored in a single edition of an ODI World Cup. He has been named Player of the Tournament at global events on three occasions:...
3. origin=local score=0.1441
   url=not included in hybrid result shape; inspect Oracle provenance
   snippet=Kohli's junior cricket career kicked off in October 2002 at the Luhnu Cricket Ground against Himachal P

The expected behavior is:

- First call: if Oracle has no fresh matching rows, Tavily is called and results are written into Oracle.
- Second call: if the written rows are still inside the TTL and score threshold, Oracle returns local results and Tavily is skipped.
- Expired or low-scoring rows trigger a refresh path through Tavily.

Because this is a persistent database demo, behavior can depend on what is already in your Oracle table. The `origin` field is the most direct signal: `local` indicates Oracle memory/cache, while `foreign` indicates Tavily supplied the result during that call.

# Mode 3: Cache Then Memory Search

`retrieval_mode="cache_then_memory"` is the layered agent-memory workflow:

1. Check fresh cache rows inside the TTL window.
2. If cache misses, check durable Oracle memory rows without the TTL filter.
3. If memory misses too, call Tavily and optionally persist fresh results.

`persistence_depth="cache_only"` means a row is intended only for cache reuse. `persistence_depth="cache_plus_memory"` means a row can serve the cache layer while fresh and remain available to the memory layer later. The optional lifecycle columns make that behavior visible in SQL Developer.

This is the main mode to show when explaining “Tavily as freshness layer, Oracle as memory layer.”

In [16]:
cache_memory_results = cache_memory_client.search(
    CACHE_QUERY,
    max_results=MAX_RESULTS,
    max_local=MAX_LOCAL,
    max_foreign=MAX_FOREIGN,
    save_foreign=True,
    **TAVILY_SEARCH_OPTIONS,
)

summarize_results(cache_memory_results, title="Cache-then-memory results")
cache_memory_origins = Counter(result.get("origin") for result in cache_memory_results)
print("Behavior:", "local cache/memory hit" if cache_memory_origins.get("local") else "cache and memory miss with Tavily fallback")
print("Cache threshold:", CACHE_SCORE_THRESHOLD)
print("Memory threshold:", MEMORY_SCORE_THRESHOLD)
print("Memory max results:", MEMORY_MAX_RESULTS)

Cache-then-memory results
-------------------------
result_count=3 origins={'local': 3}
1. origin=local score=0.3518
   url=not included in hybrid result shape; inspect Oracle provenance
   snippet=7. What are Virat Kohli’s records as captain of India? Virat Kohli is one of India’s most successful cricket captains. Major captaincy records include: • Most Test wins as Indian captain (40+) • Led India to No.1 position in ICC Test...
2. origin=local score=0.1851
   url=not included in hybrid result shape; inspect Oracle provenance
   snippet=Kohli holds several cricket records, including the most individual hundreds in ODI matches and the most runs scored in a single edition of an ODI World Cup. He has been named Player of the Tournament at global events on three occasions:...
3. origin=local score=0.1441
   url=not included in hybrid result shape; inspect Oracle provenance
   snippet=Kohli's junior cricket career kicked off in October 2002 at the Luhnu Cricket Ground against Himachal Pra

## Inspect Cache and Memory Lifecycle Metadata

This query shows the rows that make the cache/memory behavior inspectable in SQL Developer. The important columns are:

- `MEMORY_SCOPE`: `cache_only` or `cache_plus_memory`
- `EXPIRES_AT`: when the cache copy should be considered expired
- `LAST_SEEN_AT`: initial insertion timestamp for the row
- `QUERY_COUNT`: starts at 1 and increments on matched Oracle upserts when the column is included
- `SOURCE_URL` and `CONTENT_HASH`: optional stable keys for Oracle `MERGE` persistence

In [17]:
with connection.cursor() as cursor:
    cursor.execute(
        f"""
        SELECT MEMORY_SCOPE,
               EXPIRES_AT,
               LAST_SEEN_AT,
               QUERY_COUNT,
               SOURCE_URL,
               CONTENT_HASH,
               DBMS_LOB.SUBSTR({CONTENT_FIELD}, 160, 1) AS CONTENT_SNIPPET
        FROM {TABLE_NAME}
        WHERE PROVIDER_NAME = :provider_name
        ORDER BY LAST_SEEN_AT DESC NULLS LAST, RETRIEVAL_TIMESTAMP DESC NULLS LAST
        FETCH FIRST 8 ROWS ONLY
        """,
        provider_name="tavily",
    )
    lifecycle_rows = [
        {
            "memory_scope": row[0],
            "expires_at": row[1],
            "last_seen_at": row[2],
            "query_count": row[3],
            "source_url": row[4],
            "content_hash": row[5],
            "content_snippet": row[6],
        }
        for row in cursor.fetchall()
    ]

display_rows(lifecycle_rows, title="Oracle cache/memory lifecycle rows")

Oracle cache/memory lifecycle rows


[{'memory_scope': 'cache_plus_memory',
  'expires_at': datetime.datetime(2026, 6, 2, 22, 25, 45),
  'last_seen_at': datetime.datetime(2026, 6, 1, 22, 25, 45),
  'query_count': 1,
  'source_url': 'https://en.wikipedia.org/wiki/Virat_Kohli',
  'content_hash': None,
  'content_snippet': "Kohli's junior cricket career kicked off in October 2002 at the Luhnu Cricket Ground against Himachal Pradesh. His first half-century in domestic cricket happen"},
 {'memory_scope': 'cache_plus_memory',
  'expires_at': datetime.datetime(2026, 6, 2, 22, 25, 45),
  'last_seen_at': datetime.datetime(2026, 6, 1, 22, 25, 45),
  'query_count': 1,
  'source_url': 'https://www.britannica.com/biography/Virat-Kohli',
  'content_hash': None,
  'content_snippet': 'Kohli holds several cricket records, including the most individual hundreds in ODI matches and the most runs scored in a single edition of an ODI World Cup. He '},
 {'memory_scope': 'cache_plus_memory',
  'expires_at': datetime.datetime(2026, 6, 2, 22, 25, 

## Compare All Three Retrieval Modes

| Feature | Hybrid Search Mode | Freshness Cache Mode | Cache Then Memory Mode |
|--------|--------------------|----------------------|------------------------|
| Retrieval mode value | `hybrid_search` | `freshness_cache` | `cache_then_memory` |
| Main purpose | Semantic memory retrieval + fresh Tavily results | Freshness-aware result reuse | Fresh cache first, durable memory second, Tavily last |
| Local Oracle layers | Memory search | Fresh cache only | Fresh cache + durable memory |
| Uses VECTOR | Yes | Yes for local lookup | Yes for cache and memory lookup |
| Uses TTL | No for the default local memory lookup | Yes | Yes for cache phase only |
| Calls Tavily | Yes when `max_foreign > 0`; in this implementation it calls Tavily whenever `max_foreign > 0` | Yes on cache miss when `max_foreign > 0` | Yes only when both cache and memory miss |
| Best for | Agent memory/RAG where fresh search should be merged with memory | Reducing repeated API calls for recent queries | Agent workflows that should prefer cache, then memory, then freshness |
| Oracle role | Long-term semantic memory | Persistent freshness cache | Persistent cache plus durable memory |

Use hybrid search when the application wants to combine durable Oracle memory with fresh Tavily retrieval and rerank the combined set. Use freshness-cache mode when the application wants an Oracle-first TTL check and only wants Tavily when cached knowledge is missing, stale, or not relevant enough. Use cache-then-memory when the application wants to avoid Tavily if either fresh cache or durable memory can answer well enough.

In [18]:
hybrid_client.enable_native_hybrid_search = False
cache_client.enable_native_hybrid_search = False
cache_memory_client.enable_native_hybrid_search = False

print("Disabled Oracle Text native hybrid search for demo stability.")

Disabled Oracle Text native hybrid search for demo stability.


In [19]:
def retrieve_context_for_agent(question, client=hybrid_client, max_results=4):
    results = client.search(
        question,
        max_results=max_results,
        max_local=max_results,
        max_foreign=max_results,
        save_foreign=True,
        **TAVILY_SEARCH_OPTIONS,
    )
    context_blocks = []
    for index, result in enumerate(results, start=1):
        snippet = shorten(str(result.get("content", "")).replace("\n", " "), width=700, placeholder="...")
        context_blocks.append(
            f"[{index}] origin={result.get('origin')} score={result.get('score')}\n{snippet}"
        )
    return {
        "question": question,
        "context": "\n\n".join(context_blocks),
        "results": results,
    }

agent_retrieval = retrieve_context_for_agent(
    "How can Oracle VECTOR and Tavily help an AI agent keep search memory fresh?"
)

print(agent_retrieval["context"])

[1] origin=foreign score=0.9998523
anthropic gemini a local model the Oracle memory layer does not care which one you pick. Then the conversation history table, plain SQL, session ID, ro content, time stamp. Now the three memory functions and these are the heart of the whole thing. Save memory writes every message to the history table. If it's from the user, it also embeds the text and stores the vector in Oracle. One function, two writes, one database. Load memory fetches recent conversation history for a session. This gives the agent short-term context and search memory runs semantic similarity search across all stored embeddings. This is what made session 3 work. Different word, same meaning. Oracle found the...

[2] origin=foreign score=0.99980444
## Key Takeaways A unified memory core combines episodic, lexical, semantic, and relationship-aware retrieval in one governed platform. Hybrid retrieval (Oracle Text + vector + metadata filters) improves reliability in enterprise queries.

## Agentic AI Use Case

This architecture maps naturally to AI agents and RAG applications:

- Agents can search once and remember useful external knowledge in Oracle.
- Future similar questions can retrieve context from Oracle memory.
- Tavily keeps answers fresh when Oracle memory is missing or stale.
- Provenance helps agents cite sources and explain where context came from.
- Semantic deduplication keeps memory from filling with repeated results.
- Hybrid retrieval improves recall by combining semantic similarity, optional text scoring, and fresh web search.

The function below is intentionally framework-free. It shows the minimal pattern an agent can use: receive a question, retrieve context through the Oracle-backed `TavilyHybridClient`, and return compact context for a downstream model or tool.

## Cleanup / Optional Reset

The demo intentionally leaves data in Oracle so repeated notebook runs can show memory reuse.

There are now two cleanup paths:

- Manual cleanup: call `cleanup_cache()` when you want to remove expired cache rows now.
- Automatic retention: initialize the client with `auto_cleanup_cache=True`; the client checks cleanup before search and runs it only after `cache_cleanup_interval_seconds` has elapsed.

With lifecycle metadata columns present, cleanup deletes expired `cache_only` rows and preserves `cache_plus_memory` rows. That keeps durable memory from being deleted just because its cache window expired.

The destructive reset is still disabled by default. It does not drop the table or indexes automatically.

In [20]:
# Optional manual expired-cache cleanup. Review before running.
# deleted_rows = cache_client.cleanup_cache()
# print("Deleted expired cache rows:", deleted_rows)

# Optional full demo-row reset. Review before running.
# with connection.cursor() as cursor:
#     cursor.execute(
#         f"DELETE FROM {TABLE_NAME} WHERE PROVIDER_NAME = :provider_name",
#         provider_name="tavily",
#     )
# connection.commit()
# print("Deleted Tavily demo rows from", TABLE_NAME)

# Optional connection close when you are done with the notebook.
# connection.close()

## Summary

This notebook demonstrated the Oracle-only Tavily integration implemented in this repository.

Key takeaways:

- Tavily provides fresh web search.
- Oracle provides persistent AI memory.
- Hybrid search enables semantic recall through Oracle `VECTOR` plus fresh Tavily retrieval.
- Freshness-cache mode reduces repeated external calls by respecting TTL and score thresholds.
- Cache-then-memory mode checks fresh cache, then durable memory, then Tavily.
- Oracle lifecycle columns make cache vs memory behavior visible and auditable.
- URL/content-hash upsert can replace append-only persistence with Oracle `MERGE`.
- Persistence limits and score thresholds keep the memory table from growing too quickly.
- Manual and automatic cache cleanup remove expired cache rows while preserving durable memory rows.
- Convenience connection parameters are available when an application does not want to create the Oracle/Mongo handle itself.
- Oracle `VECTOR` supports similarity search over persisted Tavily results.
- Oracle JSON stores raw Tavily payloads and provenance for auditability.
- Semantic deduplication keeps memory cleaner by skipping near-duplicate inserts.
- Vector indexes such as HNSW/IVF improve performance as the memory store grows.
- The design is suitable for enterprise AI agents and RAG workflows that need freshness, durability, provenance, and recall.